# 02 - Feature Engineering

This notebook documents the feature layer used by the model: deterministic customer intelligence fields, high-cardinality business-code encoding, and leakage-safe validation design.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Build Deterministic Customer Features

In [2]:
from insurance_propensity.config import DATA_RAW_DIR, TARGET_COLUMN
from insurance_propensity.data.validation import load_raw_datasets
from insurance_propensity.features.engineering import InsuranceFeatureBuilder, CrossValidatedTargetEncoder

bundle = load_raw_datasets(DATA_RAW_DIR)
train = bundle.train
x = train.drop(columns=[TARGET_COLUMN])
y = train[TARGET_COLUMN]

builder = InsuranceFeatureBuilder().fit(x)
feature_frame = builder.transform(x.head(10_000))
feature_frame[
    [
        "Age",
        "Annual_Premium",
        "Vehicle_Age",
        "Vehicle_Damage",
        "age_group",
        "premium_band",
        "customer_risk_segment",
        "cross_sell_opportunity_segment",
    ]
].head(10)

,Age,Annual_Premium,Vehicle_Age,Vehicle_Damage,age_group,premium_band,customer_risk_segment,cross_sell_opportunity_segment
0,44,40454.0,> 2 Years,Yes,36-45,Upper Core,Protection Gap with Damage History,Priority
1,76,33536.0,1-2 Year,No,66+,Core,Uninsured Multi-Year Vehicle,Nurture
2,47,38294.0,> 2 Years,Yes,46-55,Upper Core,Protection Gap with Damage History,Priority
3,21,28619.0,< 1 Year,No,18-25,Core,Existing Vehicle Cover Signal,Low
4,29,27496.0,< 1 Year,No,26-35,Core,Existing Vehicle Cover Signal,Low
5,24,2630.0,< 1 Year,Yes,18-25,Low,Protection Gap with Damage History,High
6,23,23367.0,< 1 Year,Yes,18-25,Core,Protection Gap with Damage History,High
7,56,32031.0,1-2 Year,Yes,56-65,Core,Protection Gap with Damage History,Priority
8,24,27619.0,< 1 Year,No,18-25,Core,Existing Vehicle Cover Signal,Low
9,32,28771.0,< 1 Year,No,26-35,Core,Existing Vehicle Cover Signal,Low


## Validate Engineered Segment Coverage

In [3]:
(
    feature_frame["cross_sell_opportunity_segment"]
    .value_counts(normalize=True)
    .rename_axis("segment")
    .reset_index(name="share")
    .assign(share=lambda frame: frame["share"].round(4))
)

,segment,share
0,Low,0.4177
1,Priority,0.3072
2,High,0.1810
3,Nurture,0.0680
4,Monitor,0.0261


## Out-of-Fold Target Encoding

In [4]:
encoder = CrossValidatedTargetEncoder(columns=["Region_Code", "Policy_Sales_Channel"])
encoded_sample = encoder.fit_transform(feature_frame, y.head(10_000))
encoded_sample[["Region_Code_response_rate_te", "Policy_Sales_Channel_response_rate_te"]].describe()

,Region_Code_response_rate_te,Policy_Sales_Channel_response_rate_te
count,10000.000000,10000.000000
mean,0.127416,0.122765
std,0.049484,0.088910
min,0.034639,0.014329
25%,0.094861,0.026184
50%,0.108186,0.156722
75%,0.192620,0.204826
max,0.214163,0.358027


## Modeling Feature Set

In [5]:
from insurance_propensity.models.modeling import FEATURE_COLUMNS

feature_inventory = {
    "total_features": len(FEATURE_COLUMNS),
    "target_encoded_features": [column for column in FEATURE_COLUMNS if column.endswith("_response_rate_te")],
    "business_segments": [column for column in FEATURE_COLUMNS if column.endswith("_segment")],
}
feature_inventory

{'total_features': 21,
 'target_encoded_features': ['Region_Code_response_rate_te',
  'Policy_Sales_Channel_response_rate_te'],
 'business_segments': ['customer_risk_segment',
  'cross_sell_opportunity_segment']}